# Exercise 2: Creating and Populating Graphs in Neo4j - Design Patterns and Refactoring

In this exercise, we move beyond basic graph creation and explore **how query patterns influence your graph design**. Unlike relational databases where the schema is fixed before application development, graph database design is iterative and driven by the questions you want to answer. Key concepts we will review include:
1. **Graph Data Modeling**: Understanding nodes, relationships, and properties
2. **Design Patterns**: Common patterns for different use cases
3. **Query-Driven Design**: How your queries should shape your model
4. **Refactoring**: Improving your graph as requirements change
5. **Relationship Types**: Why specifying relationship types matters

By the end of this exercise, you'll understand how to design scalable graph databases that perform efficiently for your specific use cases.

## Step 1: Setup and Connection

Let's establish our connection to Neo4j and prepare for a deeper dive into graph modeling.

In [ ]:
%%capture
# Install required packages
%pip install neo4j networkx matplotlib pandas numpy

# Import required libraries
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
import time
import pandas as pd
import json
from datetime import datetime, timedelta
import random

# Driver setup - connect to Neo4j instance
URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "password")

# Create driver
driver = GraphDatabase.driver(URI, auth=AUTH)

# Test connection
try:
    with driver.session() as session:
        session.run("RETURN 1")
    print("Successfully connected to Neo4j")
except ServiceUnavailable:
    print("Could not connect to Neo4j. Make sure the service is running.")
    driver.close()

## Step 2: Understanding Graph Data Model Design Principles

Before we build a graph, let's understand the key design principles.

First, let's consider the key elements of a graph: **Nodes vs Properties**
- **Nodes**: Used for things you want to query, aggregate, or connect. Nodes are "things in the real world" generally.
- **Relationships**: Are the connections between those nodes - how they interact and are related to one another.
- **Properties**: Use for descriptive attributes about those "things in the real world" you are studying.

When working with nodes, relationships, and properties, be explicit: use descriptive relationship types or components (not generic "HAS" or "IS"). This helps make queries more human readable: `(customer)-[:PURCHASED]->(product)` is better than `(customer)-[:RELATED]->(product)`.

Next, when you design your queries, ask yourself: "What questions do I want to answer?". This will help you structure your graph to make those queries efficient. The right model depends on your access patterns, not universal rules. As you may know from Neo4j guidance or other lessons; it helps to think through the steps:

1) Focus on the application you're building
2) Prioritize representative queries 
3) Determine the level of denormalization 



Depending on the answers to the above investigations, you may consider some **Common Design Patterns**:
- **Simple Patterns**: With central nodes connected to many others 
- **Highly normalized graphs**: 
- **Intermediate Nodes**: Create explicit nodes for junction objects (e.g., Order between Customer and Product)
- **Denormalized Graphs**: Use relationships to express parent-child hierarchies
- **Batch Patterns**:

Let's explore these through a real example.

## Step 3: Preparation - Clear and Load ACME Corporation Data

We'll use the ACME Corporation dataset (customers, products, purchases, ratings) to demonstrate different graph design approaches and how to refactor when requirements change.

In [ ]:
# Clear all existing data
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    print("✅ Cleared all nodes and relationships")

In [ ]:
# Load sample ACME Corporation data
acme_data = {
    'customers': [
        {'id': 'C001', 'name': 'Alice Smith', 'email': 'alice@company.com', 'status': 'gold'},
        {'id': 'C002', 'name': 'Bob Johnson', 'email': 'bob@company.com', 'status': 'silver'},
        {'id': 'C003', 'name': 'Charlie Lee', 'email': 'charlie@company.com', 'status': 'bronze'},
        {'id': 'C004', 'name': 'Diana Prince', 'email': 'diana@company.com', 'status': 'gold'},
        {'id': 'C005', 'name': 'Eve Martinez', 'email': 'eve@company.com', 'status': 'silver'},
    ],
    'products': [
        {'id': 'P001', 'name': 'Small Gear', 'category': 'Gears', 'type': 'small', 'price': 49.99},
        {'id': 'P002', 'name': 'Large Gear', 'category': 'Gears', 'type': 'large', 'price': 149.99},
        {'id': 'P003', 'name': 'Bearing', 'category': 'Components', 'type': 'bearing', 'price': 29.99},
        {'id': 'P004', 'name': 'Shaft', 'category': 'Components', 'type': 'shaft', 'price': 39.99},
    ],
    'purchases': [
        {'customer_id': 'C001', 'product_id': 'P001', 'date': '2024-01-15', 'qty': 5},
        {'customer_id': 'C001', 'product_id': 'P003', 'date': '2024-01-20', 'qty': 10},
        {'customer_id': 'C002', 'product_id': 'P002', 'date': '2024-01-18', 'qty': 2},
        {'customer_id': 'C002', 'product_id': 'P004', 'date': '2024-02-01', 'qty': 8},
        {'customer_id': 'C003', 'product_id': 'P001', 'date': '2024-02-05', 'qty': 3},
        {'customer_id': 'C004', 'product_id': 'P001', 'date': '2024-02-10', 'qty': 7},
        {'customer_id': 'C004', 'product_id': 'P002', 'date': '2024-02-12', 'qty': 4},
        {'customer_id': 'C005', 'product_id': 'P003', 'date': '2024-02-15', 'qty': 6},
    ],
    'ratings': [
        {'customer_id': 'C001', 'product_id': 'P001', 'score': 5, 'text': 'Excellent quality'},
        {'customer_id': 'C001', 'product_id': 'P003', 'score': 4, 'text': 'Good product'},
        {'customer_id': 'C002', 'product_id': 'P002', 'score': 5, 'text': 'Perfect fit'},
        {'customer_id': 'C003', 'product_id': 'P001', 'score': 3, 'text': 'Average'},
        {'customer_id': 'C004', 'product_id': 'P001', 'score': 5, 'text': 'Great!'},
        {'customer_id': 'C004', 'product_id': 'P002', 'score': 4, 'text': 'Very good'},
    ]
}

print("✅ ACME Corporation data loaded:")
print(f"   Customers: {len(acme_data['customers'])}")
print(f"   Products: {len(acme_data['products'])}")
print(f"   Purchases: {len(acme_data['purchases'])}")
print(f"   Ratings: {len(acme_data['ratings'])}")

## Step 4: Design Pattern 1 - Simple Model (Basic Approach)

Let's start with a straightforward graph model:
- **Customers** connect to **Products** via PURCHASED relationships
- **Customers** rate **Products** via RATED relationships

This works well for basic queries but has limitations for complex analytics. Let's implement it and understand its strengths and weaknesses.

In [ ]:
def create_simple_model(driver, data):
    """
    Model 1: Simple direct connections
    Customers <-[PURCHASED]-> Products
    Customers <-[RATED]-> Products
    """
    ### BEGIN SOLUTION
    with driver.session() as session:
        # 1. Create Customer nodes
        for customer in data['customers']:
            session.run(
                "CREATE (c:Customer {id: $id, name: $name, email: $email, status: $status})",
                id=customer['id'], name=customer['name'],
                email=customer['email'], status=customer['status']
            )

        # 2. Create Product nodes
        for product in data['products']:
            session.run(
                "CREATE (p:Product {id: $id, name: $name, category: $category, type: $type, price: $price})",
                id=product['id'], name=product['name'],
                category=product['category'], type=product['type'], price=product['price']
            )

        # 3. Create PURCHASED relationships with date and quantity
        for purchase in data['purchases']:
            session.run("""
                MATCH (c:Customer {id: $customer_id}), (p:Product {id: $product_id})
                CREATE (c)-[:PURCHASED {date: $date, quantity: $qty}]->(p)
            """, customer_id=purchase['customer_id'], product_id=purchase['product_id'],
                date=purchase['date'], qty=purchase['qty'])

        # 4. Create RATED relationships with score and text
        for rating in data['ratings']:
            session.run("""
                MATCH (c:Customer {id: $customer_id}), (p:Product {id: $product_id})
                CREATE (c)-[:RATED {score: $score, text: $text}]->(p)
            """, customer_id=rating['customer_id'], product_id=rating['product_id'],
                score=rating['score'], text=rating['text'])

        # Print summary
        result = session.run("MATCH (n) RETURN labels(n)[0] AS type, count(*) AS count ORDER BY type")
        print("✅ Simple model created:")
        for record in result:
            print(f"   {record['type']}: {record['count']}")

        result = session.run("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS count ORDER BY type")
        print("   Relationships:")
        for record in result:
            print(f"   {record['type']}: {record['count']}")
    ### END SOLUTION

# Create the simple model
create_simple_model(driver, acme_data)

### Analysis of Simple Model

This system is the most basic example of a graph data structure. It's straightforward. 

**Strengths:**
- Simple to understand and implement
- Great for "Who has purchased what?" queries
- Can embed relationship data (date, quantity, score)

However - while it's simplicity makes it great for *some* use cases and data structures, it's not a one size fits all solution for graph data modeling. It has it's challenges.

**Weaknesses:**
- Hard to analyze trends by category (need to traverse through products)
- Difficult to find products bought together frequently
- No explicit representation of categories or customer segments
- Can't efficiently aggregate by category without intermediate nodes

## Step 5: Design Pattern 2 - Enhanced Model (Hub Pattern)

Now let's refactor by adding **Category** nodes and **CustomerSegment** nodes. This is the "Hub Node Pattern" - categorical nodes that serve as connection points.

**Why this matters for queries:**
- "Show me top products in the Gears category" - direct to category node
- "Find gold customers" - query by segment
- "What categories interest gold customers most?" - pattern matching becomes efficient

In [ ]:
def create_enhanced_model(driver, data):
    """
    Model 2: Enhanced model with hub nodes
    Categories and CustomerSegments serve as "hub" nodes
    Products -[:BELONGS_TO]-> Category
    Customers -[:HAS_SEGMENT]-> CustomerSegment
    """
    ### BEGIN SOLUTION
    with driver.session() as session:
        # 1. Extract unique categories from products
        categories = set(p['category'] for p in data['products'])

        # 2. Extract unique customer segments (status values) from customers
        segments = set(c['status'] for c in data['customers'])

        # 3. Create Category hub nodes
        for category in categories:
            session.run("CREATE (c:Category {name: $name})", name=category)

        # 4. Create CustomerSegment hub nodes
        for segment in segments:
            session.run("CREATE (s:CustomerSegment {name: $name})", name=segment)

        # 5. Link Products to Categories with BELONGS_TO relationships
        for product in data['products']:
            session.run("""
                MATCH (p:Product {id: $id}), (c:Category {name: $category})
                CREATE (p)-[:BELONGS_TO]->(c)
            """, id=product['id'], category=product['category'])

        # 6. Link Customers to CustomerSegments with HAS_SEGMENT relationships
        for customer in data['customers']:
            session.run("""
                MATCH (c:Customer {id: $id}), (s:CustomerSegment {name: $status})
                CREATE (c)-[:HAS_SEGMENT]->(s)
            """, id=customer['id'], status=customer['status'])

        print("✅ Enhanced model with hub nodes created")
        print(f"   Categories: {categories}")
        print(f"   Segments: {segments}")
    ### END SOLUTION

# Add enhanced model on top of existing simple model
create_enhanced_model(driver, acme_data)

### Analysis of Enhanced Model with Hub Nodes

**New Capabilities:**
- Can now efficiently query "What are the top-rated products in category X?"
- Can find "Which categories do gold customers prefer?"
- Hub nodes make aggregation queries much faster
- Clear separation of concerns (customers, products, categories)

**Trade-offs:**
- Slightly more complex graph structure
- More nodes to maintain
- But queries are now much more efficient for specific patterns

## Step 6: Rich Relationships - Storing Data on Graph Edges

One of the powerful features of graphs is the ability to store data on relationships, not just nodes. This lets us create intermediate nodes for complex transactions.

In [ ]:
def add_order_nodes(driver, data):
    """
    Design Pattern 3: Intermediate Nodes (Orders)
    Instead of: Customer -[PURCHASED {quantity, date}]-> Product
    Use: Customer -[PLACED]-> Order <-[CONTAINS]- Product
    
    Benefits:
    - Order becomes a first-class entity (can have relationships to other things)
    - Can track order status, fulfillment, payments
    - Can query "which orders contain multiple products?"
    """
    ### BEGIN SOLUTION
    with driver.session() as session:
        for i, purchase in enumerate(data['purchases'], 1):
            order_id = f"ORD{i:04d}"
            session.run("""
                MATCH (c:Customer {id: $customer_id}), (p:Product {id: $product_id})
                CREATE (o:Order {id: $order_id, date: $date})
                CREATE (c)-[:PLACED]->(o)
                CREATE (o)-[:CONTAINS {quantity: $qty}]->(p)
            """, customer_id=purchase['customer_id'], product_id=purchase['product_id'],
                order_id=order_id, date=purchase['date'], qty=purchase['qty'])

        print(f"✅ Created {len(data['purchases'])} Order intermediate nodes")
        result = session.run("MATCH (o:Order) RETURN count(o) AS count")
        print(f"   Total Orders in graph: {result.single()['count']}")
    ### END SOLUTION

# Add Order intermediate nodes
add_order_nodes(driver, acme_data)

### Rich Relationships Benefits

Now we can query sophisticated patterns:
- "Find customers who ordered multiple products in a single order"
- "Which products are frequently ordered together?"
- "Find the most recent order from each customer"
- "Track order status and fulfillment separately from the original purchase"

## Step 7: Demonstrating Query-Driven Design

Now let's show why our design choices matter by querying the enhanced model.

#### Task 1: Query by Category Hub

Find all products in a category and their average rating.

In [ ]:
# Efficient category query - hub pattern makes this fast
### BEGIN SOLUTION
category_query = """
MATCH (cat:Category {name: 'Gears'})<-[:BELONGS_TO]-(p:Product)
OPTIONAL MATCH (p)<-[r:RATED]-()
RETURN p.name AS product, p.price AS price,
       avg(r.score) AS avg_rating, count(r) AS num_reviews
ORDER BY avg_rating DESC
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(category_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=records[0].keys())
        print("Products in 'Gears' Category with Ratings:")
        print(df)
    else:
        print("No results - check your query")

#### Task 2: Customer Segment Analysis

Find the purchasing patterns of different customer segments.

In [ ]:
# Find which categories each segment purchases from
### BEGIN SOLUTION
segment_analysis_query = """
MATCH (seg:CustomerSegment {name: 'gold'})<-[:HAS_SEGMENT]-(c:Customer)
      -[:PLACED]->(o:Order)-[cont:CONTAINS]->(p:Product)
      -[:BELONGS_TO]->(cat:Category)
RETURN cat.name AS category,
       count(DISTINCT c) AS unique_customers,
       sum(cont.quantity) AS total_quantity
ORDER BY total_quantity DESC
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(segment_analysis_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=records[0].keys())
        print("Gold Customer Segment Analysis:")
        print(df)
    else:
        print("No results - check your query")

#### Task 3: Product Affinity (Products Bought Together)

Find products that are frequently purchased together using the Order intermediate node pattern.

In [ ]:
# Find products purchased by the same customers
### BEGIN SOLUTION
affinity_query = """
MATCH (c:Customer)-[:PURCHASED]->(p1:Product),
      (c)-[:PURCHASED]->(p2:Product)
WHERE p1.id < p2.id
RETURN p1.name AS product1, p2.name AS product2,
       count(DISTINCT c) AS shared_customers
ORDER BY shared_customers DESC
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(affinity_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=records[0].keys())
        print("Product Affinity Analysis (Bought Together):")
        print(df)
    else:
        print("No co-purchases found")

#### Task 4: Customer Lifetime Value

Calculate total spending and order count per customer using the Order and Product relationship structure.

In [ ]:
# Customer Lifetime Value calculation
### BEGIN SOLUTION
clv_query = """
MATCH (c:Customer)-[:HAS_SEGMENT]->(seg:CustomerSegment)
MATCH (c)-[:PLACED]->(o:Order)-[cont:CONTAINS]->(p:Product)
RETURN c.name AS customer, seg.name AS segment,
       count(DISTINCT o) AS num_orders,
       sum(p.price * cont.quantity) AS total_spent
ORDER BY total_spent DESC
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(clv_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=records[0].keys())
        print("Customer Lifetime Value Analysis:")
        print(df)
    else:
        print("No results")

## Step 8: Refactoring - Adding New Requirements

Real-world systems evolve. Let's add a new requirement: **Suppliers** - products have suppliers, and we want to analyze supplier relationships.

In [ ]:
def add_supplier_network(driver):
    """
    Refactoring: Add Supplier nodes and relationships
    This reduces duplication and allows supplier-based analytics
    """
    ### BEGIN SOLUTION
    suppliers = [
        {'id': 'S001', 'name': 'GearWorks Inc.'},
        {'id': 'S002', 'name': 'ComponentCo Ltd.'},
    ]

    product_suppliers = {
        'P001': 'S001', 'P002': 'S001',  # Gears from GearWorks
        'P003': 'S002', 'P004': 'S002',  # Components from ComponentCo
    }

    with driver.session() as session:
        # Create Supplier nodes
        for supplier in suppliers:
            session.run(
                "CREATE (s:Supplier {id: $id, name: $name})",
                id=supplier['id'], name=supplier['name']
            )

        # Link Products to Suppliers with SUPPLIED_BY relationships
        for product_id, supplier_id in product_suppliers.items():
            session.run("""
                MATCH (p:Product {id: $pid}), (s:Supplier {id: $sid})
                CREATE (p)-[:SUPPLIED_BY]->(s)
            """, pid=product_id, sid=supplier_id)

        print("✅ Supplier network added")
        print(f"   Suppliers: {[s['name'] for s in suppliers]}")
    ### END SOLUTION

# Add supplier network
add_supplier_network(driver)

#### Task 5: Supplier Performance Analysis

Now we can analyze supplier performance using our refactored graph.

In [ ]:
# Supplier performance query
### BEGIN SOLUTION
supplier_performance_query = """
MATCH (s:Supplier)<-[:SUPPLIED_BY]-(p:Product)
OPTIONAL MATCH (o:Order)-[cont:CONTAINS]->(p)
OPTIONAL MATCH ()-[r:RATED]->(p)
RETURN s.name AS supplier,
       count(DISTINCT p) AS product_count,
       count(DISTINCT o) AS order_count,
       avg(r.score) AS avg_rating,
       sum(cont.quantity) AS total_units_sold
ORDER BY total_units_sold DESC
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(supplier_performance_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=records[0].keys())
        print("Supplier Performance Analysis:")
        print(df)
    else:
        print("No results")

## Step 9: Testing Relationship Type Importance

Let's demonstrate why specifying relationship types matters. Bad naming makes queries unclear and maintainability suffers.

In [ ]:
# Example comparison
print("❌ BAD RELATIONSHIP NAMING:")
print("   Customer -[HAS]-> Product")
print("   Question: Does the customer own the product? Have they purchased it? Are they interested in it?")
print("   -> Ambiguous and hard to maintain\n")

print("✅ GOOD RELATIONSHIP NAMING:")
print("   Customer -[PURCHASED]-> Product  (indicates transaction)")
print("   Customer -[RATED]-> Product      (indicates review)")
print("   Customer -[INTERESTED_IN]-> Product (indicates browsing/wish list)")
print("   -> Clear intent, self-documenting, maintainable\n")

print("BEST PRACTICE: Relationship types should be VERB phrases describing the action or connection!")

## Step 10: Summary - Design Patterns Comparison

Let's visualize and compare the different models we've created.

In [ ]:
# Summary statistics of our final graph
### BEGIN SOLUTION
stats_query = """
MATCH (n)
RETURN labels(n)[0] AS node_type, count(*) AS count
ORDER BY node_type
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(stats_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=['Node Type', 'Count'])
        print("Final Graph Statistics:")
        print(df)
        print(f"\nTotal nodes: {df['Count'].sum()}")
    else:
        print("No results")

# Relationship statistics
### BEGIN SOLUTION
rel_stats_query = """
MATCH ()-[r]->()
RETURN type(r) AS rel_type, count(*) AS count
ORDER BY count DESC
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(rel_stats_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=['Relationship Type', 'Count'])
        print("\nRelationship Statistics:")
        print(df)
        print(f"Total relationships: {df['Count'].sum()}")
    else:
        print("No results")

## Summary: Key Lessons in Graph Data Modeling

### 1. **Query-Driven Design**
Your graph structure should match your query patterns. There's no universal "best" model—only the best model for your use case.

### 2. **Hub Node Pattern**
Create categorical/dimensional nodes (Categories, Segments, Suppliers) to enable efficient aggregation and filtering queries.

### 3. **Rich Relationships**
Store important data on relationships and create intermediate nodes (Orders) for complex entities.

### 4. **Relationship Type Naming**
Use verb phrases for relationship types (PURCHASED, RATED, SUPPLIED_BY). Avoid generic names like HAS or IS.

### 5. **Refactoring**
As requirements change, add new node types and relationships. The graph naturally extends without schema migrations.

### 6. **Common Patterns**
- **Hierarchies**: Use -[:PARENT]-> or -[:CHILD]-> relationships
- **Categories**: Hub nodes for grouping/filtering
- **Intermediate Nodes**: For events/transactions (Orders, Reviews)
- **Properties vs Nodes**: If you'll query/aggregate it, make it a node

### 7. **Design Questions to Ask**
- "What are my primary queries?"
- "What entities need independent querying?"
- "Are there categorical groupings I need to evolve?" (→ Use hubs)
- "Can relationships carry important metadata?" (→ Store on relationships)
- "What intermediate objects represent events?" (→ Create intermediate nodes)

By designing with these patterns in mind, you create graphs that are both performant and maintainable as your application evolves!